In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [2]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [3]:

querySalesTerritory = """
SELECT 
[TerritoryID]
      ,[Name]
      ,[CountryRegionCode]
      ,[Group]
FROM Sales.SalesTerritory
"""
tablaSalesTerritory = pd.read_sql_query(querySalesTerritory, motorBaseDatos)


queryCountryRegion = """
SELECT 
[CountryRegionCode]
      ,[Name]
FROM Person.CountryRegion
"""
tablaCountryRegion = pd.read_sql_query(queryCountryRegion, motorBaseDatos)



# tablaSalesTerritory
# tablaCountryRegion

TRANSFORMACION

In [4]:
dimensionSalesTerritory = tablaSalesTerritory.merge(tablaCountryRegion, on='CountryRegionCode')



dimensionSalesTerritory.rename(columns={
    'TerritoryID'  : 'SalesTerritoryKey',
    'Name_x' : 'SalesTerritoryRegion',
    'Name_y' : 'SalesTerritoryCountry',
    'Group' : 'SalesTerritoryGroup',
}, inplace=True)

dimensionSalesTerritory["SalesTerritoryAlternateKey"] = dimensionSalesTerritory["SalesTerritoryKey"]
dimensionSalesTerritory["SalesTerritoryImage"] = None



dimensionSalesTerritory.drop(columns={
    'CountryRegionCode'
}, inplace=True)

dimensionSalesTerritory


,SalesTerritoryKey,SalesTerritoryRegion,SalesTerritoryGroup,SalesTerritoryCountry,SalesTerritoryAlternateKey,SalesTerritoryImage
0,1,Northwest,North America,United States,1,None
1,2,Northeast,North America,United States,2,None
2,3,Central,North America,United States,3,None
3,4,Southwest,North America,United States,4,None
4,5,Southeast,North America,United States,5,None
5,6,Canada,North America,Canada,6,None
6,7,France,Europe,France,7,None
7,8,Germany,Europe,Germany,8,None
8,9,Australia,Pacific,Australia,9,None
9,10,United Kingdom,Europe,United Kingdom,10,None


CARGAR A LA BODEGA

In [5]:
dimensionSalesTerritory.to_sql('dimensionSalesTerritory',motorBodegaDatos, if_exists='replace',index=False)

10